In [6]:
import stanza
import pandas as pd 
from supar import Parser
from conllu import parse_incr
from collections import defaultdict
from pathlib import Path
from tqdm import tqdm
import re
import pandas as pd

In [2]:
def is_copular_sentence(pred_tokens):
    """
    Detect if a sentence is a COP (copula) sentence.
    
    Args:
        pred_tokens: list of dicts with keys 'form', 'upos', 'xpos', 'head', 'deprel'
        
    Returns:
        True if it's a COP sentence, else False
    """
    copula_verbs = ['be', 'is', 'am', 'are', 'was', 'were', 'being', 'been']
    copula_tokens = [t for t in pred_tokens if t["upos"] in ["AUX", "VERB"] and t["deprel"] == "cop"]
    
    if not copula_tokens:
        return False
    
    # Check if there's at least one nominal subject
    has_subject = any(t["deprel"] == "nsubj" for t in pred_tokens)
    
    return has_subject and len(copula_tokens) > 0

def is_spi(pred_tokens):
    # Find the root verb
    root_verbs = [t for t in pred_tokens if t['upos'] == 'VERB' and t['deprel'] != 'cop']
    if not root_verbs and len(root_verbs) != 1:
        return False

    # Check for a subject
    has_subject = any(t['deprel'] == 'nsubj' for t in pred_tokens)
    if not has_subject:
        return False

    # Check for a direct object
    has_obj = any(t['deprel'] in ['obj', 'dobj'] for t in pred_tokens)
    if has_obj:
        return False

    return True


def is_spt(pred_tokens):
    if not pred_tokens:
        return False

    # Find the root verb
    root_verbs = [t for t in pred_tokens if t['upos'] == 'VERB' and t['deprel'] != 'cop']
    
    if len(root_verbs) > 1:
        return False
    
    # Check for subject
    has_subject = any(t['deprel'] == 'nsubj' for t in pred_tokens)
    if not has_subject:
        return False

    # Check for direct object
    has_obj = any(t['deprel'] in ['obj', 'dobj'] for t in pred_tokens)
    if has_obj:
        return True
    return False


def is_imperative(pred_tokens):
    """
    Heuristic to detect imperative sentences without mood info.
    """
    if not pred_tokens:
        return False
    
    # Skip initial interjections or child names
    start_idx = 0
    while start_idx < len(pred_tokens) and pred_tokens[start_idx]['upos'] in ["INTJ", "PROPN"]:
        start_idx += 1
    
    if start_idx >= len(pred_tokens):
        return False
    
    first_token = pred_tokens[start_idx]
    
    # Condition 1: first meaningful token is a verb
    if first_token['upos'] not in ('VERB', "AUX"):
        return False
    
    # Condition 2: no explicit subject
    for t in pred_tokens:
        if t.get('deprel') == 'nsubj':
            return False
    
    return True

In [3]:
def categorize_sentence(pred_tokens):
    """
    Categorize a sentence based on predicted tokens with UD annotations.
    
    Args:
        pred_tokens: list of dicts, each dict has 'form', 'upos', 'xpos', 'head', 'deprel'
        
    Returns:
        category: str, one of FRA, QWH, QYN, COP, IMP, SPI, SPT, COM
    """
    
    if not pred_tokens:
        return None
    
    # Extract lists for convenience
    upos_tags = [t["upos"] for t in pred_tokens]
    xpos_tags = [t["xpos"] for t in pred_tokens]
    forms = [t["form"] for t in pred_tokens]
    
    if "VERB" not in upos_tags:
        return "FRA"
    
    if forms[-1] == "?" and upos_tags[0] == "PRON" and xpos_tags[0] == "WP":
        return "QWH"

    if forms[-1] == "?" and upos_tags[0] in ["VERB", "AUX"]:
        return "QYN"

    if is_copular_sentence(pred_tokens):
        return "COP"
    
    if is_imperative(pred_tokens):
        return "IMP"

    lexical_verbs = [t for t in pred_tokens if t['upos'] == 'VERB' and t['deprel'] not in ('aux', 'cop')]

    # COM: two or more lexical verbs
    if len(lexical_verbs) >= 2:
        return "COM"

    if len(lexical_verbs) == 1:
        has_subject = any(t['deprel'] in ('nsubj', 'nsubj:pass') for t in pred_tokens)
        if not has_subject:
            return "UNCATEGORIZED"
        
        has_obj = any(t['deprel'] in ['obj', 'dobj'] for t in pred_tokens)

        if has_obj:
            return "SPT"
        else:
            return "SPI"

    # If none of the above
    return "UNCATEGORIZED"

In [ ]:
test_path = "./UD_English-CHILDES/en_childes-ud-test.conllu"
pos_tagger_model = "./saved_models/pos/en_childes_charlm_tagger.pt"
parser_model = "./saved_models/depparse/en_childes_charlm_parser.pt"

nlp_childes = stanza.Pipeline(
        lang='en',
        processors='tokenize,pos,lemma,depparse',
        use_gpu=True,
        pos_model_path=pos_tagger_model,
        depparse_model_path=parser_model
    )

In [10]:
def load_conllu_gold(path):
    """
    Load gold annotations from a CONLLU file.
    Returns list of sentences, each as a list of dicts:
    {
        'form': str, 
        'upos': str, 
        'xpos': str, 
        'head': int, 
        'deprel': str
    }
    """
    sentences = []
    with open(path, "r", encoding="utf-8") as f:
        for tokenlist in parse_incr(f):
            tokens = []
            for token in tokenlist:
                if isinstance(token["id"], tuple):  # skip multi-word tokens like "2-3"
                    continue
                tokens.append({
                    "form": token["form"],
                    "upos": token["upostag"],
                    "xpos": token["xpostag"],
                    "head": token["head"],
                    "deprel": token["deprel"]
                })
            if tokens:
                sentences.append(tokens)
    return sentences

In [13]:
gold_sentences = load_conllu_gold(test_path)

In [ ]:
categorized_sentences = []

for gold_tokens in tqdm(gold_sentences, desc="Categorizing sentences"):
    gold_text = " ".join(t["form"] for t in gold_tokens)
    doc = nlp_childes(gold_text)
    
    pred_tokens = []
    for sent in doc.sentences:
        for word in sent.words:
            pred_tokens.append({
                "form": word.text,
                "upos": word.upos,
                "xpos": word.xpos,
                "head": int(word.head) if getattr(word, "head", None) is not None else 0,
                "deprel": word.deprel
            })
    
    category = categorize_sentence(pred_tokens)
    
    categorized_sentences.append({
        "sentence": gold_text,
        "category": category
    })

Categorizing sentences: 100%|██████████| 9591/9591 [12:14<00:00, 13.05it/s] 


In [ ]:
uncat_sentences = [s for s in categorized_sentences if s["category"] == "UNCATEGORIZED"]
uncat_df = pd.DataFrame(uncat_sentences)
uncat_df.to_csv("constructions/uncategorized.csv", index=False)

In [ ]:
fra_sentences = [s for s in categorized_sentences if s["category"] == "FRA"]
fra_df = pd.DataFrame(fra_sentences)
fra_df.to_csv("constructions/fra.csv", index=False)

In [ ]:
qwh_sentences = [s for s in categorized_sentences if s["category"] == "QWH"]
qwh_df = pd.DataFrame(qwh_sentences)
qwh_df.to_csv("constructions/qwh.csv", index=False)

In [ ]:
imp_sentences = [s for s in categorized_sentences if s["category"] == "IMP"]
imp_df = pd.DataFrame(imp_sentences)
imp_df.to_csv("constructions/imp.csv", index=False)

In [ ]:
cop_sentences = [s for s in categorized_sentences if s["category"] == "COP"]
cop_df = pd.DataFrame(cop_sentences)
cop_df.to_csv("constructions/cop.csv", index=False)

In [ ]:
spt_sentences = [s for s in categorized_sentences if s["category"] == "SPT"]
spt_df = pd.DataFrame(spt_sentences)
spt_df.to_csv("constructions/spt.csv", index=False)

In [ ]:
spi_sentences = [s for s in categorized_sentences if s["category"] == "SPI"]
spi_df = pd.DataFrame(spi_sentences)
spi_df.to_csv("constructions/spi.csv", index=False)

In [ ]:
com_sentences = [s for s in categorized_sentences if s["category"] == "COM"]
com_df = pd.DataFrame(com_sentences)
com_df.to_csv("constructions/com.csv", index=False)

In [ ]:
qyn_sentences = [s for s in categorized_sentences if s["category"] == "QYN"]
qyn_df = pd.DataFrame(qyn_sentences)
qyn_df.to_csv("constructions/qyn.csv", index=False)

## ERROR ANALYSIS 

In [ ]:
# Path to your log file
log_file = "./parser_predictions_Stanza_Custom.conllu"  
# Output file for sentences with errors
output_file = "stanza_custom_errors_only.txt"  

def parse_log_file(log_path):
    """Parse the custom Stanza log file into a list of sentences."""
    sentences = []
    current_sent = []
    with open(log_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line.startswith("# sentence:"):
                if current_sent:
                    sentences.append(current_sent)
                    current_sent = []
                current_sent.append({"meta": line})
            elif line.startswith("Gold:"):
                gold_info = {}
                for kv in line.replace("Gold: ", "").split(", "):
                    if "=" not in kv:
                        continue  # skip malformed parts
                    k, v = kv.split("=", 1)
                    gold_info[k] = v
                current_sent.append({"gold": gold_info})
            elif line.startswith("Pred:"):
                pred_info = {}
                for kv in line.replace("Pred: ", "").split(", "):
                    if "=" not in kv:
                        continue
                    k, v = kv.split("=", 1)
                    pred_info[k] = v
                if current_sent and "gold" in current_sent[-1]:
                    current_sent[-1]["pred"] = pred_info
        if current_sent:
            sentences.append(current_sent)
    return sentences


def has_error(sent_tokens):
    """Return True if the sentence has at least one POS/head/label error."""
    for token in sent_tokens[1:]:  # skip meta
        gold = token.get("gold", {})
        pred = token.get("pred", {})
        if not gold or not pred:
            continue
        if gold.get("upos") != pred.get("upos"):
            return True
        if gold.get("head") != pred.get("head"):
            return True
        if gold.get("label") != pred.get("label"):
            return True
    return False

def write_errors(sentences, out_file):
    """Write sentences with errors to a new file."""
    with open(out_file, "w", encoding="utf-8") as f:
        for sent in sentences:
            if has_error(sent):
                for token in sent:
                    if "meta" in token:
                        f.write(token["meta"] + "\n")
                    elif "gold" in token and "pred" in token:
                        f.write(
                            f"Gold: {', '.join(f'{k}={v}' for k,v in token['gold'].items())}\n"
                        )
                        f.write(
                            f"Pred: {', '.join(f'{k}={v}' for k,v in token['pred'].items())}\n"
                        )
                f.write("-" * 60 + "\n\n")

sentences = parse_log_file(log_file)
write_errors(sentences, output_file)

print(f"Saved sentences with errors to {output_file}")


Saved sentences with errors to stanza_custom_errors_only.txt
